In [1]:
import os
os.chdir("../")
os.getcwd()

'/Users/rdiya/my-personal-projects/Kidney-Disease-Classification-End-To-End-DL-Project'

In [7]:
from cnnClassifier.utils.common import read_yaml, create_directories
from pathlib import Path

CONFIG_FILE_PATH = Path("config/config.yaml")
config = read_yaml(CONFIG_FILE_PATH)
config

[2026-06-29 20:43:12,135:INFO:common:yaml file: config/config.yaml loaded successfully]


ConfigBox({'data_ingestion': {'root_dir': 'artifacts/data_ingestion', 'source': 'nazmul0087/ct-kidney-dataset-normal-cyst-tumor-and-stone', 'local_data_file': 'artifacts/data_ingestion/data.zip', 'unzip_dir': 'artifacts/data_ingestion'}})

In [8]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source: str
    local_data_file: Path
    unzip_dir: Path

In [9]:
class ConfigurationManager:
    def __init__(self, config_filepath=Path("config/config.yaml")):
        self.config = read_yaml(config_filepath)
        create_directories([self.config.data_ingestion.root_dir])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])

        return DataIngestionConfig(
            root_dir=Path(config.root_dir),
            source=config.source,
            local_data_file=Path(config.local_data_file),
            unzip_dir=Path(config.unzip_dir),
        )

In [13]:
import os
import zipfile
import kagglehub

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            dataset_path = kagglehub.dataset_download(self.config.source)
            print(f"Dataset downloaded to: {dataset_path}")
            # kagglehub downloads and extracts automatically, copy to our local path
            import shutil
            shutil.make_archive(
                str(self.config.local_data_file).replace(".zip", ""),
                'zip',
                dataset_path
            )
            print(f"Archived dataset to: {self.config.local_data_file}")
        else:
            print(f"File already exists: {self.config.local_data_file}")

    def extract_zip_file(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
        print(f"Extracted to: {unzip_path}")

In [12]:
try:
    config_manager = ConfigurationManager()
    data_ingestion_config = config_manager.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-06-29 20:44:32,700:INFO:common:yaml file: config/config.yaml loaded successfully]
[2026-06-29 20:44:32,706:INFO:common:created directory at: artifacts/data_ingestion]
[2026-06-29 20:44:32,707:INFO:common:created directory at: artifacts/data_ingestion]


100%|██████████| 1.52G/1.52G [04:28<00:00, 6.06MB/s]

Extracting files...


Dataset downloaded to: /Users/rdiya/.cache/kagglehub/datasets/nazmul0087/ct-kidney-dataset-normal-cyst-tumor-and-stone/versions/1
Archived dataset to: artifacts/data_ingestion/data.zip
Extracted to: artifacts/data_ingestion
